In [68]:
import numpy as np
import pandas as pd
from utils import *
import matplotlib.pyplot as plt

In [69]:
train = pd.read_csv('../Data/train_imputed.csv')
oot1 = pd.read_csv('../Data/oot1_imputed.csv')
oot2 = pd.read_csv('../Data/oot2_imputed.csv')

print(train.shape[0], oot1.shape[0], oot2.shape[0])

475389 174229 183730


In [70]:
flag_cols_to_keep = [c for c in train.columns if 'flag' in c]

In [71]:
def flag(series, condition):
    return condition.astype(float).where(series.notna())

def divide_positive(numerator, denominator):
    with np.errstate(divide="ignore", invalid="ignore", over="ignore"):
        ratio = numerator / denominator.where(denominator > 0)
    return ratio.replace([np.inf, -np.inf], np.nan)

def divide_all(numerator, denominator):
    with np.errstate(divide="ignore", invalid="ignore", over="ignore"):
        ratio = numerator / denominator
    return ratio.replace([np.inf, -np.inf], np.nan)

In [72]:
# temp = pd.read_csv('../Data/train-2026_with_target_variable.csv')

In [73]:
# temp.query('id == 111590592 and stmt_date == "2008-12-31"')['debt_st']

In [74]:
# train[train['debt_st'] < 0].iloc[0]

In [75]:
# train[train['debt_st'] > 0].iloc[0]

In [125]:
def build_features(df, flag_cols_to_keep):
    # 0. Identifiers and target
    identifier_features = ["id", "stmt_date", "data_type", "default", "ateco_sector"]
    features = df[identifier_features + flag_cols_to_keep].copy()

    # 1. Firm Size
    features["log_assets"] = np.log(df["asst_tot"].where(df["asst_tot"] > 0))

    # 2. Profitability
    features["net_profit_to_assets"] = divide_positive(df["profit"], df["asst_tot"])
    features["operating_margin"] = divide_positive(df["prof_operations"], df["rev_operating"])
    features["ebitda_to_assets"] = divide_positive(df["ebitda"], df["asst_tot"])
    features["roa"] = df["roa"]  # Operating profit / total assets * 100

    # 3. Liquidity and Working Capital
    features["working_capital_to_assets"] = divide_positive(df["wc_net"], df["asst_tot"])
    features["cash_to_assets"] = divide_positive(df["cash_and_equiv"], df["asst_tot"])
    features["cash_to_debt_st"] = divide_positive(df["cash_and_equiv"], df["debt_st"])
    features["current_assets_to_asset_total"] = divide_positive(df["asst_current"], df["asst_tot"])
    features["current_assets_to_debt_st"] = divide_positive(df["asst_current"], df["debt_st"])

    # 4. Solvency and Capital Structure
    debt = df["debt_st"] + df["debt_lt"]

    features["equity_to_assets"] = divide_positive(df["eqty_tot"], df["asst_tot"])
    features["debt_to_assets"] = divide_positive(debt, df["asst_tot"])
    features["debt_st_to_assets"] = divide_positive(df["debt_st"], df["asst_tot"])
    features["st_debt_share"] = divide_positive(df["debt_st"], debt)
    features["bank_debt_concentration"] = divide_positive(df["debt_bank_st"] + df["debt_bank_lt"], debt)

    # 5. Debt-Servicing Capacity
    features["interest_coverage_proxy"] = divide_positive(df["prof_operations"], df["exp_financing"])
    features["cash_flow_to_debt"] = divide_positive(df["cf_operations"], debt)
    features["debt_to_ebitda"] = divide_positive(debt, df["ebitda"])

    # 6. Cash Flow and Cash Generation
    liabilities_tot = df["asst_tot"] - df["eqty_tot"]
        
    features["operating_cashflow_to_assets"] = divide_positive(df["cf_operations"], df["asst_tot"])
    features["operating_cashflow_to_liabilities"] = divide_positive(df["cf_operations"], liabilities_tot)
    features["cashflow_profit_gap_to_assets"] = divide_positive(df["profit"] - df["cf_operations"], df["asst_tot"])

    # 7. Operational Efficiency and Receivables
    features["sales_to_assets"] = divide_positive(df["rev_operating"], df["asst_tot"])
    features["receivables_to_assets"] = divide_positive(df["AR"], df["asst_tot"])
    features["account_payables_to_assets"] = divide_positive(df["AP_st"], df["asst_tot"])

    # 8. Financial Distress and Special-State Indicators
    features["negative_equity_flag"] = flag(df["eqty_tot"], df["eqty_tot"] < 0)
    features["negative_debt_flag"] = flag(debt, debt < 0)
    features["operating_loss_flag"] = flag(df["prof_operations"], df["prof_operations"] < 0)
    features["net_loss_flag"] = flag(df["profit"], df["profit"] < 0)
    features["nonpositive_revenue_flag"] = flag(df["rev_operating"], df["rev_operating"] <= 0)
    features["zero_financing_expense_flag"] = flag(df["exp_financing"], df["exp_financing"] == 0)
    features["nonpositive_ebitda_flag"] = flag(df["ebitda"], df["ebitda"] <= 0)

    # 9. Business Characteristics and Categorical Features
    legal_structs = ["SAA", "SAU", "SPA", "SRL", "SRS", "SRU"]
    legal_feats = pd.get_dummies(df['legal_struct'].isin(legal_structs).astype(pd.CategoricalDtype(legal_structs)), prefix='legal_struct', dtype=int)
    features = pd.concat([features, legal_feats], axis=1)

    # 10. Final Feature List
    categorical_features = [c for c in features.columns if 'legal_struct' in c]
    flag_features = [c for c in features.columns if 'flag' in c]
    numerical_features = [c for c in features.columns if c not in categorical_features + identifier_features + flag_features]
    model_features = {
        'identifier_features': identifier_features,
        'numerical_features': numerical_features,
        'categorical_features': categorical_features,
        'flag_features': flag_features
    }

    return features, model_features

In [126]:
train_feats, feature_dict_train = build_features(train, flag_cols_to_keep)
oot1_feats, feature_dict_oot1 = build_features(oot1, flag_cols_to_keep)
oot2_feats, feature_dict_oot2 = build_features(oot2, flag_cols_to_keep)

assert(feature_dict_oot1 == feature_dict_train == feature_dict_oot2)

In [127]:
print('='*5,'TRAIN','='*5)
print(train_feats.isna().sum()[train_feats.isna().sum() > 0])
print('='*5,'OOT1','='*5)
print(oot1_feats.isna().sum()[oot1_feats.isna().sum() > 0])
print('='*5,'OOT2','='*5)
print(oot2_feats.isna().sum()[oot2_feats.isna().sum() > 0])

===== TRAIN =====
operating_margin                     15583
cash_to_debt_st                        952
current_assets_to_debt_st              952
st_debt_share                          189
bank_debt_concentration                189
interest_coverage_proxy              15469
cash_flow_to_debt                      189
debt_to_ebitda                       73229
operating_cashflow_to_liabilities       93
dtype: int64
===== OOT1 =====
operating_margin                      6418
cash_to_debt_st                        376
current_assets_to_debt_st              376
st_debt_share                           64
bank_debt_concentration                 64
interest_coverage_proxy               6645
cash_flow_to_debt                       64
debt_to_ebitda                       28884
operating_cashflow_to_liabilities       44
dtype: int64
===== OOT2 =====
operating_margin                      5990
cash_to_debt_st                        368
current_assets_to_debt_st              368
st_debt_share      

In [128]:
train_feats.to_csv('../Data/train_features.csv', index=False)
oot1_feats.to_csv('../Data/oot1_features.csv', index=False)
oot2_feats.to_csv('../Data/oot2_features.csv', index=False)

In [ ]:
import pickle

pickle.dump(feature_dict_train, open('../Files/all_features.pkl', 'wb'))